In [1]:
import os
import shutil

import h5py
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

from scipy.ndimage import zoom, gaussian_filter
import matplotlib.colors as mcolors
from matplotlib.colors import LogNorm, PowerNorm

from scipy.signal import medfilt2d

import sys
sys.path.append(r"C:\Users\jose1\OneDrive\PostDoc_FZU\DATA_analysis\VNA_data\VNA_analysis")
from Misc_Functions import *

In [2]:
#WORKS
#v3.1
#This version allows one to recover raw data
#We add a fucntion that recovers the raw data by adaing the reference measurment to each S value reported in the h5 file.

#v3:
# We use generic delta_mij() to remove backgrounds.
#Included posibility to calculate permeability with get_permeabilities()
#This requires Susceptibility_Methods (path to be specified, see above)
#This new functionality requires: average_permitivity (unit less) of covered waveguide, sample_lenght (in mm)

#v2.5:Experimental!!!!!
#Improved background removal when using avarage background (single_ref = False)
#Corrected final result of delta_S(), Re and Im where interchanged

#v2: Corrected names of plots
#v1: No longer Requires running ConvertH5_vKonstanz.ipynb before for the same folder or the data in format *.h5 

#Enter the folder where data is in "address" (do not erase the r)
#Give a "device" name for the naming of the files (all other parameters fro sensible name are taken from data)
#average_permitivity (unit less) of covered waveguide
#Sample_lenght that covers waveguide (in mm)
#"Single_ref" = True recommended. It subtracts the background using the spectra from the point "ref_index". If false calculates teh median background for substarction
#"direction" allows to especify the direction  of the applied field for file name. Could be used later for vector magnet
#"inPLane" determines if the measurements is done inP or not, and chooses the correct Magnetic field parametrization
#"ref_index" sets the field index to use as reference. E.g.: ref_index = 0 means the first field point (usually largest field)
#"contrast_amp" sets the contrast amplirtude for the slider in the 2D plots

In [3]:
#Folder where file is. Keep the r to use raw string in Windows.
address = Path(r"C:\Users\jose1\OneDrive\PostDoc_FZU\DATA\joseS\ml20250815b_piece\inP_m1100\FMR_2026-04-17")

In [4]:
newPath = str(address) + "\Plots"
newfolder = Path(newPath)
newfolder.mkdir(parents=True, exist_ok=True)

In [5]:
device = r"ml20250815b_piece_inP_m1110"

In [6]:
sample_lenght = 5/1000 #in m

In [7]:
epsilon = 17 #unit less

In [8]:
direction = "inP"

In [9]:
inPlane = True

In [10]:
single_ref = True

In [11]:
ref_index = 0

In [12]:
contrast_amp = 10

In [13]:
def calcField(A, inP): #A current in ampere
    #Field for 20mmGap, from Manual (inPlane)
    field1 = 0.01018 + 0.05859*A - 0.00118*(A**2) + 2.1658E-4*(A**3) - 1.77652E-5*(A**4) + 6.35589E-7*(A**5) -1.15951E-8*(A**6) + 1.06504E-10*(A**7) - 3.91644E-13*(A**8)
    #Cal_12152025_Using_ml20240308b2_dS11_5dBm (inPlane)
    field2 = 0.00675 + 0.05052*A + 6.69033E-4*(A**2) - 4.38998E-4*(A**3) + 1.11737E-4*(A**4) - 1.45078E-5*(A**5) + 1.00456E-6*(A**6) - 3.52383E-8*(A**7) + 4.88931E-10*(A**8)
    #Cal_18122025_Using_ml20240308b2_dS11_0dBm (outPlane)
    field_outP = -4.75612131323894e-10*A**8 + 5.53240901595132e-8*A**7 - 2.44397032127303e-6*A**6 + 5.44573750852832e-5*A**5 - 0.000670050456046096*A**4 + 0.00455904626621348*A**3 - 0.0156901329345105*A**2 + 0.0762165481027225*A + 0.0164075279665365

    field_inP = (field1 + field2)/2
    
    if inP == True:
        field = field_inP
    else:
        field = field_outP
    return field #field in T

In [14]:
def load_any(f, possible_paths):
    """
    Try several dataset paths and return the first one that exists.
    Raises an error if none exist.
    """
    for p in possible_paths:
        if p in f:
            return f[p][:]
    raise KeyError(f"None of the paths exist: {possible_paths}")

In [15]:
def field_dir(h_x, h_y, h_z):
    """
    Determine which field components vary significantly, and compute
    the effective field magnitude using those components.

    Parameters
    ----------
    h_x, h_y, h_z : array-like
        Field components.

    Returns
    -------
    H : ndarray
        The effective field magnitude (only using components that vary > min_var).
    direction : str
        String containing the letters of the varying components, e.g. "x", "yz".
    """

    min_var = 5e-3  # T

    # Variation of each component
    var = [
        np.max(h_x) - np.min(h_x),
        np.max(h_y) - np.min(h_y),
        np.max(h_z) - np.min(h_z)
    ]

    # Determine which components vary significantly
    c = [v > min_var for v in var]      # booleans
    direction = "".join(d for d, flag in zip("xyz", c) if flag)

    # Compute effective field using only selected components
    H = np.sqrt(
        (c[0] * h_x)**2 +
        (c[1] * h_y)**2 +
        (c[2] * h_z)**2
    )

    return H, direction


In [16]:
def get_permeabilities(single, ref_index, S11, S21, S12, S22, freqs, epsilon, lenght):
    "Calculate parameters"
    reflexion, p = ite_reflexion_p(abs(S11), abs(S21), abs(S12), abs(S22), deem=False, deem_phase=1)
    results = ite_permitt_permeab(lenght, reflexion, p, freqs, epsilon)
    raw_permitivity = np.array(results["permittivity"])
    raw_permeability = np.array(results["first_eval_permeab"])
    permb1_R, permb1_I = delta_mij(single, results["first_eval_permeab"], results["first_eval_permeab"][ref_index])
    permb2_R, permb2_I = delta_mij(single, results["second_eval_permeab"], results["second_eval_permeab"][ref_index])

    return{
        "Permeab1_R": np.array(permb1_R), "Permeab1_I": np.array(permb1_I), "Permeab1_M": abs(np.array(permb1_R+1j*permb1_I)), 
        "Permeab2_R": np.array(permb2_R), "Permeab2_I": np.array(permb2_I), "Permeab2_M": abs(np.array(permb2_R+1j*permb2_I)),
        "Eff_permitivity_R": raw_permitivity.real, "Eff_permitivity_I": raw_permitivity.imag, 
        "Eff_permeability_R": raw_permeability.real, "Eff_permeability_I": raw_permeability.imag
    }

In [17]:
def delta_S(single, S, S_ref):
    deltaS_R, deltaS_I = delta_mij(single, S, S_ref)     
    return deltaS_R + 1j*deltaS_I

In [18]:
def get_raw_S_matrix(S11, S21, S12, S22, S_refs):
    """Sij are the full arrays for all fields (#Freqs, #fields). S_refs contains single ij arrays for a "single" field (#freqs, 1)."""
    raw_S11 = delta_S(True, S11, -S_refs["11"])
    raw_S12 = delta_S(True, S12, -S_refs["12"])
    raw_S21 = delta_S(True, S21, -S_refs["21"])
    raw_S22 = delta_S(True, S22, -S_refs["22"])

    return{
        "11": raw_S11, "21": raw_S21,
        "22": raw_S22, "12": raw_S12,
    }

In [19]:
def get_dS_matrix(Z11, Z21, Z12, Z22, single, ref_index, freqs, H):
    dS11 = delta_S(single, Z11, Z11[ref_index])
    dS12 = delta_S(single, Z12, Z12[ref_index])
    dS21 = delta_S(single, Z21, Z21[ref_index])
    dS22 = delta_S(single, Z22, Z22[ref_index])

    dS21_dH = np.gradient(dS21, H, axis=0)
    dS21_M_dH = np.gradient(abs(np.array(dS21)), H, axis=0)
        
    return{
        #"dS11R": np.array(dS11).real, "dS11I": np.array(dS11).imag,
        #"dS21R": np.array(dS21).real, "dS21I": np.array(dS21).imag,
        #"dS12R": np.array(dS12).real, "dS12I": np.array(dS12).imag,
        #"dS22R": np.array(dS22).real, "dS22I": np.array(dS22).imag,
        "dS11M": abs(np.array(dS11)), "dS21M": abs(np.array(dS21)),
        "dS12M": abs(np.array(dS12)), "dS22M": abs(np.array(dS22)),
        "D_dS21_DH_R": np.array(dS21_dH).real, "D_dS21_DH_I": np.array(dS21_dH).imag,
        "D_dS21_DH_M": abs(np.array(dS21_dH)), "D_dS21M_DH_M": abs(np.array(dS21_M_dH)),
        "D_dS21M_DH_R": np.array(dS21_M_dH).real,
        #"dS11_phi" : np.array(np.angle(dS11)), "dS21_phi" : np.array(np.angle(dS21)),
        #"dS22_phi" : np.array(np.angle(dS22)), "dS12_phi" : np.array(np.angle(dS12)),
        "D_dS21_DH_phi" : np.array(np.angle(dS21_dH))
    }

In [20]:
def plot_data_plotly(freqs, dLij, single_R,low_x, high_x, save=True, plot=True):
    freqs_ghz = freqs*1e-9
    #freqs_ghz = freqs * 1e-9  # Convert to GHz once
    n_max = dLij[ list(calcDS.keys())[0] ].shape[0] - 1  # Max value for n

    # Create figure with two subplots
    fig = make_subplots(rows=2, cols=1, shared_xaxes=False, vertical_spacing=0.1)

    # Initial plot for n=0
    def get_lines(n):
        lines = [
            #{'y': dLij["dS11R"][n], 'name': 'Re(dL11)', 'color': 'blue', 'row': 1},
            #{'y': dLij["dS11R"][n], 'name': 'Im(dL11)', 'color': 'red', 'row': 1},
            #{'y': dLij["dS22R"][n], 'name': 'Re(dL22)', 'color': 'cyan', 'row': 1},
            #{'y': dLij["dS22I"][n], 'name': 'Im(dL22)', 'color': 'magenta', 'row': 1},
            {'y': dLij["dS11M"][n], 'name': 'Mag(dS11)', 'color': 'orange', 'row': 1},
            {'y': dLij["dS22M"][n], 'name': 'Mag(dS22)', 'color': 'mediumpurple', 'row': 1},

            #{'y': dLij["dS12R"][n], 'name': 'Re(dL12)', 'color': 'blue', 'row': 2},
            #{'y': dLij["dS12I"][n], 'name': 'Im(dL12)', 'color': 'red', 'row': 2},
            #{'y': dLij["dS21R"][n], 'name': 'Re(dL21)', 'color': 'cyan', 'dash': 'dash', 'row': 2},
            #{'y': dLij["dS21I"][n], 'name': 'Im(dL21)', 'color': 'magenta', 'dash': 'dash', 'row': 2},
            {'y': dLij["dS12M"][n], 'name': 'Mag(dS12)', 'color': 'orange', 'row': 2},
            {'y': dLij["dS21M"][n], 'name': 'Mag(dS21)', 'color': 'mediumpurple', 'dash': 'dash', 'row': 2},
            {'y': dLij["D_dS21M_DH_R"][n], 'name': 'D_dS12_DH', 'color': 'blue', 'row': 2},
            {'y': dLij["Permeab1_R"][n], 'name': 'mu_1_R', 'color': 'blue', 'row': 2},
            {'y': dLij["Permeab1_I"][n], 'name': 'mu_1_I', 'color': 'red', 'row': 2},
            {'y': dLij["Permeab1_M"][n], 'name': 'mu_1_M', 'color': 'orange', 'row': 2},
            {'y': dLij["Permeab2_R"][n], 'name': 'mu_2_R', 'color': 'blue', 'row': 2},
            {'y': dLij["Permeab2_I"][n], 'name': 'mu_2_I', 'color': 'red', 'row': 2},
            {'y': dLij["Permeab2_M"][n], 'name': 'mu_2_M', 'color': 'mediumpurple', 'row': 2},
            {'y': dLij["Eff_permitivity_R"][n], 'name': 'Eff_Epsilon_R', 'color': 'mediumpurple', 'row': 2},
            {'y': dLij["Eff_permitivity_I"][n], 'name': 'Eff_Epsilon_I', 'color': 'mediumpurple', 'row': 2},
            {'y': dLij["Eff_permeability_R"][n], 'name': 'Eff_mu_R', 'color': 'mediumpurple', 'row': 2},
            {'y': dLij["Eff_permeability_I"][n], 'name': 'Eff_mu_I', 'color': 'mediumpurple', 'row': 2},
            
        ]
        return lines

    lines = get_lines(0)
    for line in lines:
        fig.add_trace(go.Scattergl(
            x=freqs_ghz,
            y=line['y'],
            mode='lines',
            name=line['name'],
            line=dict(color=line['color'], dash=line.get('dash', 'solid'))
        ), row=line['row'], col=1)

    # Update layout
    fig.update_layout(
        height=800,
        width=900,
        title= f"Interactive dSij Analysis, H(0)="+ str(int(1000*fields[0]))+ f"mT [Ref. H({ref_index})=" + str(int(fields[ref_index])) +" mT]",
        showlegend=True,
        xaxis=dict(title="f (GHz)", range=[low_x, high_x]),
        xaxis2=dict(title="f (GHz)", range=[low_x, high_x]),
        yaxis=dict(title="dSii (U)", range=[-500, 500]),
        yaxis2=dict(title="dSij (U)", range=[-150, 150]),
        sliders=[
            {
                'active': 0,
                'currentvalue': {"prefix": "n = "},
                'pad': {"t": 50},
                'steps': [
                    {
                        'label': str(n) +" (" + str(int(1000*fields[n]))+ "mT)",
                        'method': 'update',
                        'args': [
                            {'y': [line['y'] for line in get_lines(n)]},
                            {'title': f"Interactive dSij Analysis, H({n})="+ str(int(1000*fields[n]))+ "mT"}
                        ]
                    }
                    for n in range(n_max + 1)
                ]
            }
        ]
    )

    if save:
        if single_R:
            fig.write_html(newfolder / (device + f"_H_{direction}" + f"_{int(power)}dBm" + f"_dSij_vsf_refi_{ref_index}.html"))
        else:
            fig.write_html(newfolder / (device + f"_H_{direction}" + f"_{int(power)}dBm" + f"_dSij_vsf_MeanRef.html"))
    if plot:
        fig.show()

In [21]:
def smooth_data(data, sigma=1):
    return gaussian_filter(data, sigma=sigma)

def plot_dynamic_dSij(field, frequency, dLij, single_R, interp, interpQ, smooth, customz, z_min, z_max, save, plot=False):
    # Preprocess: sort by field
    field = field #T
    sorted_indices = np.argsort(field)
    sorted_field = field[sorted_indices]

    fig = go.Figure()
    buttons = []

    for i, (key, data) in enumerate(dLij.items()):
        sorted_data = data[sorted_indices]

        # Optional smoothing and interpolation
        processed_data = smooth_data(sorted_data, sigma=0.75) if smooth else sorted_data
        high_res_data = zoom(processed_data, interpQ) if interp else processed_data 
        new_frequency = np.linspace(frequency.min(), frequency.max(), high_res_data.shape[1])

        initial_key="D_dS21M_DH_M"
        visible = (key == initial_key)

        fig.add_trace(go.Heatmap(
            z=high_res_data.T,
            x=sorted_field,
            y=new_frequency,
            colorscale="RdBu_r" if "R" in key or "I" in key else "magma",
            zmin = z_min if customz else ( -100 if "R" in key or "I" in key else 1 ),
            zmax = z_max if customz else ( 100 if "R" in key or "I" in key else np.max(high_res_data) ),
            colorbar=dict(title=key),
            visible=visible,
            zsmooth='best',
            name=key
        ))

        # Add button to toggle visibility
        visibility = [False] * len(dLij)
        visibility[i] = True
        buttons.append(dict(
            label=key,
            method="update",
            args=[{"visible": visibility},
                  {"title": f"{key} Spectrum" + f" [Ref. H({ref_index})=" + str(fields[ref_index]) +" T]", "coloraxis.colorbar.title": key}]
        ))

    #Contrast slider points and limits
    steps = []
    contrast_array = np.linspace(-contrast_amp, contrast_amp, 1000) #if "R" in key or "I" in key else np.linspace(0, contrast_amp, 1000)
    for s in contrast_array :  # smaller = high contrast, larger = low contrast
        steps.append(dict(
            method="restyle",
            args=[{"zmin": -s, "zmax": s}], #if "R" in key or "I" in key else [{"zmin": 0, "zmax": s}],
            label=f"{s:.2f}"
        ))
        
    # Layout and dropdown
    fig.update_layout(
        updatemenus=[{
            "buttons": buttons,
            "direction": "down",
            "showactive": True,
            "x": 1.1,
            "xanchor": "left",
            "y": 1.15,
            "yanchor": "top"
        }],
        xaxis_title="Magnetic field (T)",
        yaxis_title="Frequency (GHz)",
        title=f"{initial_key} Spectrum" + f" [Ref. H({ref_index})=" + str(fields[ref_index]) +" T]",
        height=600,
        width=800,
        sliders=[{ #contrast slider definition
            "active": 10,
            "steps": steps,
            "x": 0.1,
            "y": -0.1,
        }]
    )

    if save:
        if single_R:
            fig.write_html(newfolder / (device + f"_H_{direction}" + f"_{int(power)}dBm" + f"_dSij_vsfvsH_refi_{ref_index}.html"))
        else:
            fig.write_html(newfolder / (device + f"_H_{direction}" + f"_{int(power)}dBm" + "_dSij_vsfvsH_MeanRef.html"))
    if plot:
        fig.show()

In [22]:
folder = os.path.abspath(address)

for fname in os.listdir(folder):
    # Process .h5 files
    if fname.endswith("h5.ma8"):
        # Open the HDF5 file
        with h5py.File(address/fname, "r") as f:
            #NumPy arrays
            S11R = f["/data/VNA d_S11R"][:]
            S11I = f["/data/VNA d_S11I"][:]
            S21R = f["/data/VNA d_S21R"][:]
            S21I = f["/data/VNA d_S21I"][:]
            S12R = f["/data/VNA d_S12R"][:]
            S12I = f["/data/VNA d_S12I"][:]
            S22R = f["/data/VNA d_S22R"][:]
            S22I = f["/data/VNA d_S22I"][:]
            freqs = f["/data/VNA frequency"][0, :] #Hz
            #fields = f["/data/magnet Field"][:] #T
            currs = f["/data/magnet current"][:] #A
            #yfield = load_any(f, ["/data/yfield", "/data/angle-yfield"]) #T
            #zfield = load_any(f, ["/data/zfield", "/data/plane-zfield"]) #T
            #temp = f["/data/T_sample"][0]
            power = f["/data/VNA power"][0]
            #Reference part
            S11R_ref = f["/data/VNA ref_S11R"][0,:]
            S11I_ref = f["/data/VNA ref_S11I"][0,:]
            S21R_ref = f["/data/VNA ref_S21R"][0,:]
            S21I_ref = f["/data/VNA ref_S21I"][0,:]
            S12R_ref = f["/data/VNA ref_S12R"][0,:]
            S12I_ref = f["/data/VNA ref_S12I"][0,:]
            S22R_ref = f["/data/VNA ref_S22R"][0,:]
            S22I_ref = f["/data/VNA ref_S22I"][0,:]
            
        S_matrix = {#"S11R": S11R, "S11I": S11I, "S21R": S21R, "S21I": S21I, 
                    "S11": S11R + 1j * S11I, "S21": S21R + 1j * S21I,
                   "S22": S22R + 1j * S22I, "S12": S12R + 1j * S12I,}
        
        S_refs = {"11": S11R_ref + 1j * S11I_ref  , "21": S21R_ref + 1j * S21I_ref , 
                 "12": S12R_ref + 1j * S12I_ref , "22": S22R_ref + 1j * S22I_ref }
        raw_S_matrix = get_raw_S_matrix(S_matrix["S11"], S_matrix["S21"], S_matrix["S12"], S_matrix["S22"], S_refs)
        
        #Determine field
        fields = calcField(currs, inPlane) #T
        field_pts = len(fields)
        ############Calculations
        calcDS = get_dS_matrix(S_matrix["S11"], S_matrix["S21"], S_matrix["S12"], S_matrix["S22"], single_ref, ref_index, freqs, fields)
        calcDS = {k: v.astype(np.float32) for k, v in calcDS.items()}#compresing data to 4 bits
        #calcdMu = get_permeabilities(single_ref, ref_index, S_matrix['S11'], S_matrix['S21'], S_matrix['S12'], S_matrix['S22'], freqs*1e-9, epsilon, sample_lenght)
        calcdMu = get_permeabilities(single_ref, ref_index, raw_S_matrix['11'], raw_S_matrix['21'], raw_S_matrix['12'], raw_S_matrix['22'], freqs*1e-9, epsilon, sample_lenght)
        calcdMu = {k: v.astype(np.float32) for k, v in calcdMu.items()}#compresing data to 4 bits
        results = calcDS | calcdMu
        plot_data_plotly(freqs, results, single_ref,low_x=5, high_x=6, save=True, plot=False)
        plot_dynamic_dSij(fields, freqs*1e-9, results, single_ref, interp=False, interpQ=1, smooth=False, customz=True, z_min=1, z_max=0.7, save=True, plot=False)

C:\Users\jose1\OneDrive\PostDoc_FZU\DATA_analysis\VNA_data\VNA_analysis\Susceptibility_Methods.py:24: RuntimeWarning: invalid value encountered in sqrt
  reflexion_1 = k + np.sqrt(k*k-1)
C:\Users\jose1\OneDrive\PostDoc_FZU\DATA_analysis\VNA_data\VNA_analysis\Susceptibility_Methods.py:25: RuntimeWarning: invalid value encountered in sqrt
  reflexion_2 = k - np.sqrt(k*k-1)
C:\Users\jose1\OneDrive\PostDoc_FZU\DATA_analysis\VNA_data\VNA_analysis\Susceptibility_Methods.py:64: RuntimeWarning: invalid value encountered in scalar divide
  ratio_1 = one_plus_r / one_minus_r
C:\Users\jose1\OneDrive\PostDoc_FZU\DATA_analysis\VNA_data\VNA_analysis\Susceptibility_Methods.py:65: RuntimeWarning: invalid value encountered in scalar divide
  ratio_2 = one_minus_r / one_plus_r
C:\Users\jose1\OneDrive\PostDoc_FZU\DATA_analysis\VNA_data\VNA_analysis\Susceptibility_Methods.py:24: RuntimeWarning:

invalid value encountered in sqrt

C:\Users\jose1\OneDrive\PostDoc_FZU\DATA_analysis\VNA_data\VNA_analysis\Susc